# General Magnetar-Driven Supernova: Flux Density and Magnitude Fitting

This notebook provides a complete tutorial for the `general_magnetar_driven_supernova` model in **redback-jax**, covering:

1. **Solver backends** — switching between `diffrax` (adaptive Tsit5, default) and `euler` (fixed-step scan)
2. **Tolerance control** — how `rtol`/`atol` affect diffrax accuracy and speed
3. **Flux density fitting** — `FluxDensityLikelihood` with nested sampling
4. **Magnitude fitting** — `Likelihood` + `general_magnetar_supernova_spectra_diffrax`
5. **Fittable SED parameters** — adding `cutoff_wavelength` and `alpha_uv` to the prior

**Requirements**  
- `redback-jax` (this repo)  
- `diffrax`, `jax >= 0.4`, `bilby`, `astropy`  
- Optional: `jax_supernovae` (magnitude fitting only — Section 4)

**Physics**  
Relativistic ejecta ODE (Lorentz factor, radius, volume, internal energy),  
TemperatureFloor photosphere (Villar+ 2017),  
CutoffBlackbody SED (Nicholl+ 2017).  
Requires float64 — enable before any JAX call.

In [ ]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore")

# Make redback_jax importable when running from examples/
_repo_root = str(pathlib.Path().absolute().parent)
if _repo_root not in sys.path:
    sys.path.insert(0, _repo_root)

import jax
jax.config.update("jax_enable_x64", True)   # required — ODE state spans >100 dex

import jax.numpy as jnp
import numpy as np
import matplotlib.pyplot as plt
import time as _time

print(f"JAX {jax.__version__}  |  devices: {jax.devices()}")

In [ ]:
from redback_jax.models import (
    general_magnetar_driven_supernova_bolometric,
    general_magnetar_driven_supernova_bolometric_and_vej,
    general_magnetar_driven_supernova,
)
from redback_jax.inference import (
    FluxDensityLikelihood, Prior, LogUniform, Uniform,
)

# Optional: magnitude fitting (requires jax_supernovae)
try:
    from redback_jax.models import general_magnetar_supernova_spectra_diffrax
    from redback_jax.inference import Likelihood
    from redback_jax.transient import Transient
    _HAS_SPECTRA = True
    print("jax_supernovae: available — magnitude fitting enabled")
except (ImportError, AttributeError):
    _HAS_SPECTRA = False
    print("jax_supernovae: not found — Section 4 (magnitude fitting) will be skipped")

from astropy.cosmology import Planck18
import astropy.constants as const

# ---------------------------------------------------------------------------
# Shared true parameters used throughout this notebook
# ---------------------------------------------------------------------------
TRUTH = dict(
    mej               = 3.5,       # solar masses
    E_sn              = 1.5e51,    # erg
    kappa             = 0.1,       # cm²/g (fixed)
    l0                = 1e44,      # erg/s
    tau_sd            = 3e6,       # s
    nn                = 3.0,       # braking index (fixed)
    kappa_gamma       = 1.0,       # cm²/g (fixed)
    temperature_floor = 4000.0,    # K
    f_nickel          = 0.0,       # (fixed)
    cutoff_wavelength = 3000.0,    # Å (fixed)
    alpha_uv          = 1.0,       # (fixed)
)

Z      = 0.05
DL_CM  = float(Planck18.luminosity_distance(Z).cgs.value)
C_CMS  = float(const.c.cgs.value)

print(f"Redshift z = {Z},  DL = {DL_CM:.3e} cm")

---
## 1. Solver Backends: `diffrax` vs `euler`

All three public functions accept a `solver` keyword:

| `solver=` | ODE backend | Extra kwargs | Notes |
|---|---|---|---|
| `'diffrax'` **(default)** | Adaptive Tsit5 (4th/5th-order RK) | `rtol`, `atol` | ~380× faster than redback |
| `'euler'` | Fixed-step Euler via `lax.scan` | `n_grid` | Deterministic, simple to debug |

Because `solver` is listed in `static_argnames`, JAX JIT compiles separate kernels for each value — there is no runtime branching overhead.

In [ ]:
# Source-frame time array
t_src = jnp.geomspace(1.0, 100., 30)   # days
base_kw = dict(
    mej=TRUTH['mej'], E_sn=TRUTH['E_sn'],
    kappa=TRUTH['kappa'], l0=TRUTH['l0'],
    tau_sd=TRUTH['tau_sd'], nn=TRUTH['nn'],
    kappa_gamma=TRUTH['kappa_gamma'],
)

# ── diffrax (default) ────────────────────────────────────────────────────────
# First call triggers JIT compilation
_t0 = _time.perf_counter()
lbol_diffrax = general_magnetar_driven_supernova_bolometric(t_src, **base_kw)
lbol_diffrax.block_until_ready()
print(f"diffrax — compile+run: {(_time.perf_counter()-_t0)*1e3:.0f} ms")

# Steady-state timing (skip first 5 calls, then average 200)
for _ in range(5):
    general_magnetar_driven_supernova_bolometric(t_src, **base_kw).block_until_ready()
ts_d = []
for _ in range(200):
    _t0 = _time.perf_counter()
    general_magnetar_driven_supernova_bolometric(t_src, **base_kw).block_until_ready()
    ts_d.append(_time.perf_counter() - _t0)
t_diffrax_ms = np.median(ts_d) * 1e3

# ── Euler (fixed-step) ───────────────────────────────────────────────────────
for _ in range(5):
    general_magnetar_driven_supernova_bolometric(
        t_src, **base_kw, solver='euler', n_grid=2000
    ).block_until_ready()
ts_e = []
for _ in range(50):
    _t0 = _time.perf_counter()
    general_magnetar_driven_supernova_bolometric(
        t_src, **base_kw, solver='euler', n_grid=2000
    ).block_until_ready()
    ts_e.append(_time.perf_counter() - _t0)
t_euler_ms = np.median(ts_e) * 1e3
lbol_euler = general_magnetar_driven_supernova_bolometric(
    t_src, **base_kw, solver='euler', n_grid=2000
)

print(f"\nSteady-state timing:")
print(f"  diffrax (default, rtol=1e-5):  {t_diffrax_ms:.3f} ms/call")
print(f"  euler   (n_grid=2000):         {t_euler_ms:.3f} ms/call")
print(f"\nMax |log10L| difference between backends: {float(jnp.max(jnp.abs(lbol_diffrax - lbol_euler))):.4f} dex")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.plot(t_src, lbol_diffrax, 'b-',  lw=2, label='diffrax (default)')
ax.plot(t_src, lbol_euler,   'r--', lw=2, label='euler (n_grid=2000)', alpha=0.8)
ax.set_xlabel('Source-frame time (days)')
ax.set_ylabel(r'$\log_{10}(L_{\rm bol}\,/\,{\rm erg\,s^{-1}})$')
ax.set_title('Bolometric light curve — both backends')
ax.legend()

ax = axes[1]
ax.plot(t_src, lbol_diffrax - lbol_euler, 'k-', lw=1.5)
ax.axhline(0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('Source-frame time (days)')
ax.set_ylabel(r'$\Delta\log_{10}L$ (diffrax − euler)')
ax.set_title('Residuals between backends')

plt.tight_layout()
plt.savefig('solver_comparison.pdf', bbox_inches='tight')
plt.show()

---
## 2. Tolerance Control: `rtol` and `atol`

The diffrax backend accepts `rtol` (relative) and `atol` (absolute) tolerances for the Tsit5 PID step-size controller.

- **Tighter tolerances** → more ODE steps → more accurate but slower  
- **Looser tolerances** → fewer ODE steps → faster but slightly less accurate  

Default values `rtol=1e-5, atol=1e-8` are a good balance for inference.  
For gradient-based optimization (HMC), tighten to `rtol=1e-7, atol=1e-10`.

In [ ]:
tolerance_configs = [
    dict(rtol=1e-3, atol=1e-6,  label='loose (1e-3/1e-6)'),
    dict(rtol=1e-5, atol=1e-8,  label='default (1e-5/1e-8)'),
    dict(rtol=1e-7, atol=1e-10, label='tight (1e-7/1e-10)'),
]

# Reference: tight solution
lbol_ref = general_magnetar_driven_supernova_bolometric(
    t_src, **base_kw, rtol=1e-10, atol=1e-12
)

print(f"{'Config':<25} {'Time (ms)':>10} {'Max error (dex)':>18}")
print('-' * 58)

tol_results = []
for cfg in tolerance_configs:
    rtol, atol, label = cfg['rtol'], cfg['atol'], cfg['label']

    # Warmup
    for _ in range(5):
        general_magnetar_driven_supernova_bolometric(
            t_src, **base_kw, rtol=rtol, atol=atol
        ).block_until_ready()
    # Time
    ts = []
    for _ in range(200):
        _t0 = _time.perf_counter()
        out = general_magnetar_driven_supernova_bolometric(
            t_src, **base_kw, rtol=rtol, atol=atol
        )
        out.block_until_ready()
        ts.append(_time.perf_counter() - _t0)

    t_ms  = np.median(ts) * 1e3
    err   = float(jnp.max(jnp.abs(out - lbol_ref)))
    tol_results.append({'label': label, 't_ms': t_ms, 'err': err, 'lbol': np.array(out)})
    print(f"  {label:<23} {t_ms:>10.3f} {err:>18.2e}")

In [ ]:
colors_tol = ['tab:orange', 'tab:blue', 'tab:green']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
for r, c in zip(tol_results, colors_tol):
    ax.plot(t_src, r['lbol'], color=c, lw=2, label=r['label'])
ax.set_xlabel('Source-frame time (days)')
ax.set_ylabel(r'$\log_{10}(L_{\rm bol})$')
ax.set_title('Effect of rtol/atol on light curve')
ax.legend(fontsize=9)

ax = axes[1]
ref = np.array(lbol_ref)
for r, c in zip(tol_results, colors_tol):
    ax.semilogy(t_src, np.abs(r['lbol'] - ref) + 1e-16,
                color=c, lw=1.5, label=r['label'])
ax.set_xlabel('Source-frame time (days)')
ax.set_ylabel(r'$|\Delta\log_{10}L|$ vs ultra-tight ref')
ax.set_title('Error vs ultra-tight reference (1e-10/1e-12)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('tolerance_comparison.pdf', bbox_inches='tight')
plt.show()

print("\nTiming summary:")
for r in tol_results:
    print(f"  {r['label']:<25}: {r['t_ms']:.3f} ms/call")

---
## 3. Synthetic Data

Generate a synthetic multiband dataset used for both the flux-density and magnitude fitting sections.  
We use 5 SDSS-like bands, 20 log-spaced epochs, and add 5% magnitude noise.

In [ ]:
# ── Observation grid ─────────────────────────────────────────────────────────
BANDS       = ['u', 'g', 'r', 'i', 'z']
BAND_COLORS = ['violet', 'royalblue', 'forestgreen', 'darkorange', 'firebrick']
BAND_CWAV   = np.array([3543., 4770., 6231., 7625., 9134.])   # Å
BAND_FREQ   = C_CMS * 1e8 / BAND_CWAV                          # Hz

N_EPOCHS    = 20
T_OBS       = np.geomspace(3., 100., N_EPOCHS)                  # observer-frame days
T_MJD_TRUE  = 58600.0                                           # assumed explosion epoch
MAG_ERR     = 0.05                                              # sigma per observation

# Interleaved layout: row i has epochs[i//5] and band[i%5]
t_all  = np.repeat(T_OBS, 5)         # shape (100,)
nu_all = np.tile(BAND_FREQ, N_EPOCHS) # shape (100,)

# ── Fixed parameters for model call ─────────────────────────────────────────
FIXED_FD = dict(
    kappa             = TRUTH['kappa'],
    nn                = TRUTH['nn'],
    kappa_gamma       = TRUTH['kappa_gamma'],
    temperature_floor = TRUTH['temperature_floor'],
    luminosity_distance = DL_CM,
    redshift          = Z,
    f_nickel          = TRUTH['f_nickel'],
    cutoff_wavelength = TRUTH['cutoff_wavelength'],
    alpha_uv          = TRUTH['alpha_uv'],
)
FREE_PARAMS = ['mej', 'E_sn', 'l0', 'tau_sd', 'temperature_floor']

# ── Generate truth flux (observer-frame, mJy) ─────────────────────────────
# The model takes observer-frame times and frequencies directly
t_jnp  = jnp.array(t_all,  dtype=jnp.float64)
nu_jnp = jnp.array(nu_all, dtype=jnp.float64)

F_true = np.array(
    general_magnetar_driven_supernova(
        t_jnp, nu_jnp,
        mej=TRUTH['mej'], E_sn=TRUTH['E_sn'], l0=TRUTH['l0'],
        tau_sd=TRUTH['tau_sd'],
        **FIXED_FD,
    )
)

# ── Add noise ────────────────────────────────────────────────────────────────
rng     = np.random.default_rng(42)
m_true  = -2.5 * np.log10(np.maximum(F_true, 1e-10) / 3.631e6)
m_noisy = m_true + rng.normal(0, MAG_ERR, size=len(m_true))
F_obs   = 3.631e6 * 10**(-0.4 * m_noisy)
F_err   = F_obs * (np.log(10) / 2.5) * MAG_ERR

print(f"Dataset: {len(F_obs)} observations ({N_EPOCHS} epochs × {len(BANDS)} bands)")
print(f"Flux range: {F_obs.min():.3e} – {F_obs.max():.3e} mJy")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(16, 3.5), sharey=False)
for b_idx, (band, color, ax) in enumerate(zip(BANDS, BAND_COLORS, axes)):
    mask = np.arange(b_idx, len(t_all), 5)
    ax.errorbar(t_all[mask], F_obs[mask], F_err[mask],
                fmt='o', color=color, ms=4, lw=0.8, label='noisy data')
    ax.plot(t_all[mask], F_true[mask], '--', color=color, lw=1.5, label='truth')
    ax.set_xlabel('Days (observer frame)')
    ax.set_ylabel('Flux density (mJy)')
    ax.set_title(f'SDSS {band}')
    ax.legend(fontsize=8)
plt.suptitle(
    f'Synthetic SDSS ugriz photometry  (z={Z}, σ={MAG_ERR} mag)',
    y=1.02, fontsize=12
)
plt.tight_layout()
plt.savefig('synthetic_data.pdf', bbox_inches='tight')
plt.show()

---
## 4. Flux Density Fitting: `FluxDensityLikelihood`

`FluxDensityLikelihood` works with any model returning **F_mJy** and requires no `jax_supernovae` dependency.  
It follows the same interface as `Likelihood` — compose with `Prior` → `NestedSampler`.

We fit 4 free parameters: `mej`, `E_sn`, `l0`, `tau_sd`.

In [ ]:
# Remove temperature_floor from FIXED_FD for this fit
# (we keep it fixed here; see Section 5 for fitting SED params)
FIXED_FD_FIT = {**FIXED_FD}  # copy

prior_fd = Prior([
    LogUniform(0.5,   20.,  name='mej'),           # M_sun
    LogUniform(1e50,  1e52, name='E_sn'),           # erg
    LogUniform(1e42,  1e46, name='l0'),             # erg/s
    LogUniform(1e4,   1e8,  name='tau_sd'),         # s
])

likelihood_fd = FluxDensityLikelihood(
    model       = general_magnetar_driven_supernova,  # uses diffrax by default
    time        = t_all,
    frequency   = nu_all,
    flux_obs    = F_obs,
    flux_err    = F_err,
    fixed_params = FIXED_FD_FIT,
)
print(likelihood_fd)

# Build and time the JIT-compiled log-likelihood
_t0 = _time.perf_counter()
log_like_fd = likelihood_fd._make_log_likelihood(prior_fd)
print(f"JIT compile: {(_time.perf_counter()-_t0)*1e3:.0f} ms")

# Sanity check at truth
theta_truth = jnp.array([TRUTH[k] for k in prior_fd.names])
logL_truth  = float(log_like_fd(theta_truth))
print(f"log L at truth = {logL_truth:.2f}  (χ²/N ≈ {-2*logL_truth/len(F_obs):.2f})")

# Steady-state timing
for _ in range(5): log_like_fd(theta_truth).block_until_ready()
ts_ll = []
for _ in range(300):
    _t0 = _time.perf_counter()
    log_like_fd(theta_truth).block_until_ready()
    ts_ll.append(_time.perf_counter() - _t0)
print(f"log-likelihood call: {np.median(ts_ll)*1e3:.3f} ms  (median 300 calls)")

In [ ]:
# ── Bilby + nestle nested sampling (nlive=150, ~1-3 min on CPU) ───────────────
try:
    import bilby
    from bilby.core.prior import LogUniform as BilbyLogUniform

    class _JAXLogLike(bilby.Likelihood):
        """Thin bilby wrapper around the JIT-compiled JAX log-likelihood."""
        def __init__(self, jax_fn, param_names):
            super().__init__(parameters={})
            self._fn    = jax_fn
            self._names = param_names
        def log_likelihood(self):
            theta = jnp.array([self.parameters[k] for k in self._names])
            return float(self._fn(theta))

    bilby_priors_fd = bilby.core.prior.PriorDict({
        'mej':    BilbyLogUniform(0.5,  20.,  latex_label=r'$M_{\rm ej}$'),
        'E_sn':   BilbyLogUniform(1e50, 1e52, latex_label=r'$E_{\rm SN}$'),
        'l0':     BilbyLogUniform(1e42, 1e46, latex_label=r'$L_0$'),
        'tau_sd': BilbyLogUniform(1e4,  1e8,  latex_label=r'$\tau_{\rm sd}$'),
    })

    print("Running bilby+nestle (nlive=150) — flux density fit ...")
    _t0_run = _time.perf_counter()
    result_fd = bilby.run_sampler(
        likelihood = _JAXLogLike(log_like_fd, prior_fd.names),
        priors     = bilby_priors_fd,
        sampler    = 'nestle',
        nlive      = 150,
        label      = 'fd_magnetar',
        outdir     = 'fit_fd_results',
        save       = True,
        verbose    = False,
    )
    t_run_fd = _time.perf_counter() - _t0_run
    print(f"Done in {t_run_fd:.0f} s  |  log Z = {result_fd.log_evidence:.2f} ± {result_fd.log_evidence_err:.2f}")
    _HAS_BILBY = True

except ImportError:
    _HAS_BILBY = False
    print("bilby not installed — skipping nested sampling run.")
    print("Install with: pip install bilby nestle")

In [ ]:
if _HAS_BILBY:
    # ── Parameter recovery table ─────────────────────────────────────────────
    print(f"\n{'Parameter':<10} {'Truth':>12} {'Median':>12} {'16th':>12} {'84th':>12} {'In 1σ':>6}")
    print('-' * 66)
    for name in prior_fd.names:
        truth = TRUTH[name]
        s     = np.array(result_fd.posterior[name])
        lo, med, hi = np.percentile(s, [16, 50, 84])
        in1 = 'Y' if lo <= truth <= hi else 'N'
        print(f"  {name:<8} {truth:>12.3e} {med:>12.3e} {lo:>12.3e} {hi:>12.3e} {in1:>6}")

    # ── Posterior light curves ────────────────────────────────────────────────
    T_DENSE = np.geomspace(2., 120., 60)
    t_d  = jnp.array(np.repeat(T_DENSE, 5),    dtype=jnp.float64)
    nu_d = jnp.array(np.tile(BAND_FREQ, 60),   dtype=jnp.float64)

    post_df = result_fd.posterior
    rng_pc  = np.random.default_rng(7)
    idx_draws = rng_pc.choice(len(post_df), size=200, replace=True)
    F_draws = np.zeros((200, len(t_d)))
    for j, i in enumerate(idx_draws):
        p = {name: float(post_df[name].iloc[i]) for name in prior_fd.names}
        F_draws[j] = np.array(
            general_magnetar_driven_supernova(t_d, nu_d, **FIXED_FD_FIT, **p)
        )

    fig, axes_lc = plt.subplots(1, 5, figsize=(16, 3.5), sharey=False)
    for b_idx, (band, color, ax) in enumerate(zip(BANDS, BAND_COLORS, axes_lc)):
        b_mask = np.arange(b_idx, len(t_d), 5)
        q16, q50, q84 = np.percentile(F_draws[:, b_mask], [16, 50, 84], axis=0)
        ax.fill_between(T_DENSE, q16, q84, color=color, alpha=0.3, label='68% CI')
        ax.plot(T_DENSE, q50, color=color, lw=1.5, label='median')
        d_mask = np.arange(b_idx, len(t_all), 5)
        ax.plot(t_all[d_mask], F_true[d_mask], 'k--', lw=1, alpha=0.6, label='truth')
        ax.errorbar(t_all[d_mask], F_obs[d_mask], F_err[d_mask],
                    fmt='o', color=color, ms=4, lw=0.8, label='data')
        ax.set_xlabel('Days')
        ax.set_ylabel('mJy')
        ax.set_title(f'SDSS {band}', fontsize=10)
        ax.legend(fontsize=7, ncol=2)
    plt.suptitle('Flux density posterior — bilby+nestle', y=1.02)
    plt.tight_layout()
    plt.savefig('fd_posterior_lcs.pdf', bbox_inches='tight')
    plt.show()

---
## 5. Magnitude Fitting: `Likelihood` + Spectra Model

`general_magnetar_supernova_spectra_diffrax` produces a full time × wavelength spectra grid via the CutoffBlackbody SED, which is then integrated through bandpass curves by `jax_supernovae`.

> **Requires** `jax_supernovae` — the cell is skipped automatically if not installed.

The API mirrors `Likelihood` for all other spectra models in redback-jax.

In [ ]:
if not _HAS_SPECTRA:
    print("Skipping magnitude fitting — jax_supernovae not available.")
    print("Install: pip install jax_supernovae")
else:
    # ── Generate synthetic magnitude data ────────────────────────────────────
    from redback_jax.sources import PrecomputedSpectraSource

    MAG_BANDS   = ['bessellb', 'bessellv', 'bessellr']   # 3 optical bands
    N_EP_MAG    = 15
    T_OBS_MAG   = np.linspace(5., 80., N_EP_MAG)

    # Evaluate spectra at truth to get magnitudes
    out_spec = general_magnetar_supernova_spectra_diffrax(
        redshift          = Z,
        lum_dist          = DL_CM,
        mej               = TRUTH['mej'],
        E_sn              = TRUTH['E_sn'],
        kappa             = TRUTH['kappa'],
        l0                = TRUTH['l0'],
        tau_sd            = TRUTH['tau_sd'],
        nn                = TRUTH['nn'],
        kappa_gamma       = TRUTH['kappa_gamma'],
        temperature_floor = TRUTH['temperature_floor'],
        f_nickel          = TRUTH['f_nickel'],
        cutoff_wavelength = TRUTH['cutoff_wavelength'],
        alpha_uv          = TRUTH['alpha_uv'],
    )
    source = PrecomputedSpectraSource(
        phases     = out_spec.time,
        wavelengths= out_spec.lambdas,
        flux_grid  = out_spec.spectra,
    )

    # Build noisy magnitude observations
    mjd_t0     = 58600.0
    obs_mjd    = T_OBS_MAG + mjd_t0
    times_list, bands_list, mags_list = [], [], []
    rng_mag = np.random.default_rng(13)
    for band in MAG_BANDS:
        true_m = np.array(source.bandmag({'amplitude': 1.0}, band, T_OBS_MAG))
        noisy  = true_m + rng_mag.normal(0, MAG_ERR, N_EP_MAG)
        times_list.extend(obs_mjd.tolist())
        bands_list.extend([band] * N_EP_MAG)
        mags_list.extend(noisy.tolist())

    transient_mag = Transient(
        time      = np.array(times_list),
        y         = np.array(mags_list),
        y_err     = np.full(len(mags_list), MAG_ERR),
        bands     = bands_list,
        data_mode = 'magnitude',
        name      = 'fake_magnetar_BVR',
        redshift  = Z,
    )
    print(f"Magnitude dataset: {len(transient_mag.time)} obs ({N_EP_MAG} epochs × {len(MAG_BANDS)} bands)")

In [ ]:
if _HAS_SPECTRA:
    FIXED_MAG = dict(
        redshift          = Z,
        lum_dist          = DL_CM,
        kappa             = TRUTH['kappa'],
        nn                = TRUTH['nn'],
        kappa_gamma       = TRUTH['kappa_gamma'],
        f_nickel          = TRUTH['f_nickel'],
        cutoff_wavelength = TRUTH['cutoff_wavelength'],
        alpha_uv          = TRUTH['alpha_uv'],
    )

    prior_mag = Prior([
        LogUniform(0.5,   20.,  name='mej'),
        LogUniform(1e50,  1e52, name='E_sn'),
        LogUniform(1e42,  1e46, name='l0'),
        LogUniform(1e4,   1e8,  name='tau_sd'),
        LogUniform(1e3,   2e4,  name='temperature_floor'),
        Uniform(58580., 58620., name='t0'),
    ])

    likelihood_mag = Likelihood(
        model        = general_magnetar_supernova_spectra_diffrax,
        transient    = transient_mag,
        fixed_params = FIXED_MAG,
        t0_key       = 't0',
        evaluation_mode = 'direct_photometry',  # fast bandpass integration
    )
    print(likelihood_mag)

    # JIT compile
    _t0 = _time.perf_counter()
    log_like_mag = likelihood_mag._make_log_likelihood(prior_mag)
    print(f"JIT compile: {(_time.perf_counter()-_t0):.1f} s")

In [ ]:
if _HAS_SPECTRA and _HAS_BILBY:
    from bilby.core.prior import Uniform as BilbyUniform

    bilby_priors_mag = bilby.core.prior.PriorDict({
        'mej':               BilbyLogUniform(0.5,  20.,  latex_label=r'$M_{\rm ej}$'),
        'E_sn':              BilbyLogUniform(1e50, 1e52, latex_label=r'$E_{\rm SN}$'),
        'l0':                BilbyLogUniform(1e42, 1e46, latex_label=r'$L_0$'),
        'tau_sd':            BilbyLogUniform(1e4,  1e8,  latex_label=r'$\tau_{\rm sd}$'),
        'temperature_floor': BilbyLogUniform(1e3,  2e4,  latex_label=r'$T_{\rm floor}$'),
        't0':                BilbyUniform(58580., 58620., latex_label=r'$t_0$'),
    })

    print("Running bilby+nestle (nlive=150) — magnitude fit ...")
    _t0_run = _time.perf_counter()
    result_mag = bilby.run_sampler(
        likelihood = _JAXLogLike(log_like_mag, prior_mag.names),
        priors     = bilby_priors_mag,
        sampler    = 'nestle',
        nlive      = 150,
        label      = 'mag_magnetar',
        outdir     = 'fit_mag_results',
        save       = True,
        verbose    = False,
    )
    t_run_mag = _time.perf_counter() - _t0_run
    print(f"Done in {t_run_mag:.0f} s  |  log Z = {result_mag.log_evidence:.2f} ± {result_mag.log_evidence_err:.2f}")

    # Parameter recovery
    TRUTH_MAG = {**TRUTH, 't0': mjd_t0}
    print(f"\n{'Parameter':<18} {'Truth':>12} {'Median':>12} {'In 1σ':>6}")
    print('-' * 54)
    for name in prior_mag.names:
        truth = TRUTH_MAG.get(name, float('nan'))
        s     = np.array(result_mag.posterior[name])
        lo, med, hi = np.percentile(s, [16, 50, 84])
        in1 = 'Y' if lo <= truth <= hi else 'N'
        print(f"  {name:<16} {truth:>12.3e} {med:>12.3e} {in1:>6}")

---
## 6. Fittable SED Parameters: `cutoff_wavelength` and `alpha_uv`

Two SED parameters can be added to the prior and inferred jointly with the physical parameters:

| Parameter | Default | Description |
|---|---|---|
| `cutoff_wavelength` | 3000 Å | UV cutoff wavelength; below this the SED is suppressed |
| `alpha_uv` ∈ [0, 4) | 1.0 | UV power-law slope: SED ∝ (λ/λ_c)^α below λ_c; α=0 → plain Planck, α=1 → Nicholl+17 |

The normalization integral over the blue side is solved analytically via the upper incomplete gamma function, so both parameters are differentiable (compatible with HMC).

In [ ]:
# ── Demonstrate alpha_uv and cutoff_wavelength effect on the SED ─────────────
from redback_jax.sed import cutoff_blackbody_flux_density

# Photosphere conditions at one epoch
lbol_val = 1e44   # erg/s
T_val    = 8000.  # K
r_val    = 3e14   # cm

# Wavelength grid 1000–10000 Å → frequency
lam_ang  = np.geomspace(1000., 10000., 300)
nu_arr   = C_CMS * 1e8 / lam_ang

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Panel A: vary alpha_uv
ax = axes[0]
for alpha in [0.0, 0.5, 1.0, 1.5, 2.0]:
    F = cutoff_blackbody_flux_density(
        jnp.array(nu_arr, dtype=jnp.float64),
        jnp.full(300, lbol_val, dtype=jnp.float64),
        jnp.full(300, T_val,    dtype=jnp.float64),
        jnp.full(300, r_val,    dtype=jnp.float64),
        DL_CM,
        cutoff_wavelength_ang=3000.0,
        alpha_uv=alpha,
    )
    ax.plot(lam_ang, np.array(F), label=f'α={alpha}')
ax.axvline(3000., color='k', ls='--', lw=1, label='λ_c = 3000 Å')
ax.set_xlabel('Wavelength (Å)')
ax.set_ylabel('Flux density (mJy)')
ax.set_title('Vary α_UV (λ_c = 3000 Å fixed)')
ax.legend(fontsize=9)

# Panel B: vary cutoff_wavelength
ax = axes[1]
for lc in [1500., 2000., 3000., 4000., 5000.]:
    F = cutoff_blackbody_flux_density(
        jnp.array(nu_arr, dtype=jnp.float64),
        jnp.full(300, lbol_val, dtype=jnp.float64),
        jnp.full(300, T_val,    dtype=jnp.float64),
        jnp.full(300, r_val,    dtype=jnp.float64),
        DL_CM,
        cutoff_wavelength_ang=lc,
        alpha_uv=1.0,
    )
    ax.plot(lam_ang, np.array(F), label=f'λ_c = {int(lc)} Å')
ax.set_xlabel('Wavelength (Å)')
ax.set_ylabel('Flux density (mJy)')
ax.set_title('Vary λ_c (α_UV = 1.0 fixed)')
ax.legend(fontsize=9)

plt.suptitle('CutoffBlackbody SED — fittable SED parameters', y=1.02)
plt.tight_layout()
plt.savefig('sed_parameters.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# ── Example: adding cutoff_wavelength and alpha_uv to the prior ──────────────
# Simply include them in Prior — they are passed as keyword arguments to the model

prior_sed = Prior([
    LogUniform(0.5,   20.,  name='mej'),
    LogUniform(1e50,  1e52, name='E_sn'),
    LogUniform(1e42,  1e46, name='l0'),
    LogUniform(1e4,   1e8,  name='tau_sd'),
    LogUniform(1500., 8000., name='cutoff_wavelength'),  # <-- SED parameter in prior
    Uniform(0.,       3.,   name='alpha_uv'),            # <-- SED parameter in prior
])

# FIXED params no longer include cutoff_wavelength / alpha_uv
FIXED_SED = {k: v for k, v in FIXED_FD_FIT.items()
             if k not in ('cutoff_wavelength', 'alpha_uv')}

likelihood_sed = FluxDensityLikelihood(
    model        = general_magnetar_driven_supernova,
    time         = t_all,
    frequency    = nu_all,
    flux_obs     = F_obs,
    flux_err     = F_err,
    fixed_params = FIXED_SED,
)

_t0 = _time.perf_counter()
log_like_sed = likelihood_sed._make_log_likelihood(prior_sed)
print(f"JIT compile: {(_time.perf_counter()-_t0)*1e3:.0f} ms")

theta_sed_truth = jnp.array([
    TRUTH['mej'], TRUTH['E_sn'], TRUTH['l0'], TRUTH['tau_sd'],
    TRUTH['cutoff_wavelength'], TRUTH['alpha_uv'],
])
logL_sed = float(log_like_sed(theta_sed_truth))
print(f"log L at truth (with SED params free) = {logL_sed:.2f}")
print("\nPrior:")
print(prior_sed)
print("\nFixed:")
for k, v in FIXED_SED.items():
    print(f"  {k} = {v}")

---
## 7. Summary

### Solver selection

```python
# Default (adaptive Tsit5, ~380x faster than redback)
lbol = general_magnetar_driven_supernova_bolometric(t, **params)

# Explicit diffrax with custom tolerances
lbol = general_magnetar_driven_supernova_bolometric(t, **params,
           solver='diffrax', rtol=1e-7, atol=1e-10)

# Fixed-step Euler (deterministic, 2000-point grid)
lbol = general_magnetar_driven_supernova_bolometric(t, **params,
           solver='euler', n_grid=2000)
```

### Flux density fitting

```python
likelihood = FluxDensityLikelihood(
    model=general_magnetar_driven_supernova,
    time=t_obs, frequency=nu_obs,
    flux_obs=F_obs, flux_err=F_err,
    fixed_params={...},
)
```

### Magnitude fitting

```python
likelihood = Likelihood(
    model=general_magnetar_supernova_spectra_diffrax,
    transient=transient,
    fixed_params={...},
    t0_key='t0',
    evaluation_mode='direct_photometry',
)
```

### SED parameters in prior

```python
prior = Prior([
    ...,
    LogUniform(1500., 8000., name='cutoff_wavelength'),
    Uniform(0., 3., name='alpha_uv'),
])
# Remove them from fixed_params — they flow automatically into the model.
```

### References

- Sarin et al. 2022, [MNRAS 516 4949](https://ui.adsabs.harvard.edu/abs/2022MNRAS.516.4949S)
- Omand & Sarin 2024, [MNRAS 527 6455](https://ui.adsabs.harvard.edu/abs/2024MNRAS.527.6455O)
- Nicholl et al. 2017, [ApJ 850 55](https://ui.adsabs.harvard.edu/abs/2017ApJ...850...55N)
- Villar et al. 2017, [ApJ 851 L21](https://ui.adsabs.harvard.edu/abs/2017ApJ...851L..21V)